In [2]:
from process_mobidb_scores import get_mobidb_scores
from alphasync_data import get_alphasync_data, parse_alphasync_data
from protein_region_features import features_binding_region_df, get_amino_acid_composition, calculate_scd, calculate_shd, calculate_fcr, get_sec_str_freq, get_average_feature_value
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# get the current directory
current_dir = Path.cwd()

# path to the input fasta file
input_fasta = current_dir.parents[1] / "data" / "raw" / "mobidb_search_2026-05-05T08-28-46.fasta"

# path to the processed data folder
processed_data_folder = current_dir.parents[1] / "data" / "processed"

# define the output data folder for the alphasync data
output_data_folder = current_dir.parents[1] / "data" / "raw" / "alphasync_data"



In [4]:
# get the mobidb scores for the given input fasta file and the given score patterns
human_proteome_seq_df = get_mobidb_scores(str(input_fasta), ["derived-binding_mode_disorder_to_disorder-mobi", "derived-binding_mode_disorder_to_order-mobi", "derived-binding_mode_context_dependent-mobi"])

# print the first 5 rows of the complete data dataframe
print(human_proteome_seq_df.head())

# print the number of rows in the complete data dataframe
print(f"Number of rows in the complete data dataframe: {human_proteome_seq_df.shape[0]}")

  uniprot_id                                           sequence  \
0     P31946  MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...   
1     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
2     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
3     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
4     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
0                                                NaN   
1  0000000000000000000000000000000000000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                                NaN   
4  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
0                                                NaN   
1                                                NaN   
2  1111111111111111111111111111111100000000000000...   
3  000000000000000000000000000000000

In [5]:
# filter the dataframe to get only those that has mobidb scores for at least one of the binding mode scores
binding_mode_scores = ["derived_binding_mode_disorder_to_disorder_mobi", "derived_binding_mode_disorder_to_order_mobi", "derived_binding_mode_context_dependent_mobi"]
df_binding_idr_seq = human_proteome_seq_df.dropna(subset=binding_mode_scores, how="all").copy()

# print the number of rows in the filtered dataframe
print(f"Number of rows in the filtered dataframe: {df_binding_idr_seq.shape[0]}")

print(df_binding_idr_seq.head())

Number of rows in the filtered dataframe: 4751
  uniprot_id                                           sequence  \
1     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
2     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
3     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
4     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   
5     P27348  MEKTELIQKAKLAEQAERYDDMATCMKAVTEQGAELSNEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
1  0000000000000000000000000000000000000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                                NaN   
4  0000000000000000000000000000000000000000000000...   
5  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
1                                                NaN   
2  1111111111111111111111111111111100000000000000...   
3  000000000000000000000000000000000000000000

In [9]:
# save the dataframe as csv files in the processed data folder
df_binding_idr_seq.to_csv(processed_data_folder / "binding_idr_sequences.csv", index=False)

In [6]:
# get the alphasync data for the binding idr sequences
print("Downloading the alphasync data for the given uniprot IDs...")
get_alphasync_data(df_binding_idr_seq["uniprot_id"].tolist(), str(output_data_folder))

Failed to download A0A0A0MSA1 (Status: 404)
Failed to download F2Z2U4 (Status: 404)
Failed to download A0A8V8TRG9 (Status: 404)
Failed to download E9PG32 (Status: 404)
Failed to download A0A669KB38 (Status: 404)


In [6]:
# remove the ids that couldn't be downloaded from the alphasync data and save the filtered dataframe as a csv file in the processed data folder
remove_ids = ["A0A0A0MSA1", "F2Z2U4", "E9PG32", "A0A8V8TRG9", "A0A669KB38"]

# filter the dataframe to remove the rows with the uniprot ids in the remove_ids list
df_binding_idr_seq_filtered = df_binding_idr_seq[~df_binding_idr_seq["uniprot_id"].isin(remove_ids)].copy()
#df_binding_idr_seq_filtered.to_csv(processed_data_folder / "binding_idr_sequences_filtered.csv", index=False)

In [ ]:
# read the filtered dataframe from the csv file in the processed data folder
#df_binding_idr_seq_filtered = pd.read_csv(processed_data_folder / "binding_idr_sequences_filtered.csv")

In [7]:
# add the alphasync data for the filtered dataframe
df_binding_idr_seq_filtered["plddt"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "plddt", str(output_data_folder), divide_by_100=True))
df_binding_idr_seq_filtered["relasa"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "relasa", str(output_data_folder)))
df_binding_idr_seq_filtered["sec"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "sec", str(output_data_folder)))

print(df_binding_idr_seq_filtered.head())

# save the updated dataframe as csv and pickle files in the processed data folder
#df_binding_idr_seq_filtered.to_csv(processed_data_folder / "binding_idr_sequences_filtered_with_alphasync.csv", index=False)
df_binding_idr_seq_filtered.to_pickle(processed_data_folder / "binding_idr_sequences_filtered_with_alphasync.pkl")

  uniprot_id                                           sequence  \
1     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
2     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
3     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
4     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   
5     P27348  MEKTELIQKAKLAEQAERYDDMATCMKAVTEQGAELSNEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
1  0000000000000000000000000000000000000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                                NaN   
4  0000000000000000000000000000000000000000000000...   
5  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
1                                                NaN   
2  1111111111111111111111111111111100000000000000...   
3  0000000000000000000000000000000000000000000000...   
4                                   

In [8]:
# load the binding region dataframes from the pickle files in the processed data folder
df_binding_idr_seq_filtered = pd.read_pickle(processed_data_folder / "binding_idr_sequences_filtered_with_alphasync.pkl")

In [9]:

# get features for the binding regions from that of the complete protein sequence
df_binding_region_do = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_disorder_to_order_mobi", {"plddt", "relasa", "sec"})
df_binding_region_do["binding_mode"] = "disorder_to_order"
print(df_binding_region_do.head())
df_binding_region_dd = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_disorder_to_disorder_mobi", {"plddt", "relasa", "sec"})
df_binding_region_dd["binding_mode"] = "disorder_to_disorder"
print(df_binding_region_dd.head())
df_binding_region_cd = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_context_dependent_mobi", {"plddt", "relasa", "sec"})
df_binding_region_cd["binding_mode"] = "context_dependent"
print(df_binding_region_cd.head())


# save the dataframes as csv files and pickle files in the processed data folder
df_binding_region_do.to_csv(processed_data_folder / "binding_region_disorder_to_order.csv", index=False)
df_binding_region_do.to_pickle(processed_data_folder / "binding_region_disorder_to_order.pkl")
df_binding_region_dd.to_csv(processed_data_folder / "binding_region_disorder_to_disorder.csv", index=False)
df_binding_region_dd.to_pickle(processed_data_folder / "binding_region_disorder_to_disorder.pkl")
df_binding_region_cd.to_csv(processed_data_folder / "binding_region_context_dependent.csv", index=False)
df_binding_region_cd.to_pickle(processed_data_folder / "binding_region_context_dependent.pkl")

  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                                sec  \
0       32  [C, C, H, H, H, H, H, H, H, H, H, H, H, H, H, ...   
1      214         [H, H, H, H, H, H, T, G, G, G, C, C, H, H]   
2       67  [C, C, P, P, P, P, P, P, P, G, G, G, S, C, H, ...   
3      449  [H, H, H, H, H, H, H, H, H, H, H, H, H, H, H, ...   
4      104  [C, C, C, C, C, C, C, C, C, C, C, C, C, C, C, ...   

                                              relasa  \
0  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2 

In [4]:
# read the dataframes from the pickle files in the processed data folder
df_binding_region_do = pd.read_pickle(processed_data_folder / "binding_region_disorder_to_order.pkl")
df_binding_region_dd = pd.read_pickle(processed_data_folder / "binding_region_disorder_to_disorder.pkl")
df_binding_region_cd = pd.read_pickle(processed_data_folder / "binding_region_context_dependent.pkl")

# combine the three dataframes into one dataframe
df_binding_region_combined = pd.concat([df_binding_region_do, df_binding_region_dd, df_binding_region_cd], ignore_index=True)

In [5]:
# filter the dataframe to remove rows where the plddt column is empty or not a list
is_not_empty = df_binding_region_combined["plddt"].apply(lambda x: isinstance(x, list) and len(x) > 0)
df_binding_region_combined = df_binding_region_combined[is_not_empty]

In [ ]:
#df_binding_region_combined["amino_acid_composition"] = df_binding_region_combined["binding_region_seq"].apply(get_amino_acid_composition)

#df_expanded = pd.DataFrame(df_binding_region_combined["amino_acid_composition"].tolist(), index=df_binding_region_combined.index).add_prefix("aa_freq_")
#df_expanded

,aa_freq_A,aa_freq_C,aa_freq_D,aa_freq_E,aa_freq_F,aa_freq_G,aa_freq_H,aa_freq_I,aa_freq_K,aa_freq_L,aa_freq_M,aa_freq_N,aa_freq_P,aa_freq_Q,aa_freq_R,aa_freq_S,aa_freq_T,aa_freq_V,aa_freq_W,aa_freq_Y
0,0.1250,0.0000,0.1250,0.1562,0.0000,0.0312,0.0000,0.0000,0.0938,0.0625,0.0938,0.0000,0.0000,0.0625,0.0625,0.0312,0.0000,0.0938,0.0000,0.0625
1,0.1429,0.0000,0.2857,0.1429,0.0714,0.0000,0.0000,0.0714,0.0000,0.1429,0.0000,0.0714,0.0000,0.0000,0.0000,0.0000,0.0714,0.0000,0.0000,0.0000
2,0.1111,0.0000,0.0556,0.1111,0.0000,0.0000,0.0556,0.0000,0.0556,0.1667,0.0000,0.0556,0.1111,0.1667,0.0000,0.0556,0.0556,0.0000,0.0000,0.0000
3,0.0476,0.0000,0.0952,0.1429,0.0476,0.0000,0.0000,0.0000,0.1905,0.1429,0.0000,0.0000,0.0000,0.0476,0.0952,0.0952,0.0476,0.0000,0.0000,0.0476
4,0.0000,0.0000,0.0000,0.0455,0.0227,0.0455,0.0000,0.0455,0.1364,0.0909,0.0000,0.0682,0.1136,0.0909,0.0909,0.1591,0.0455,0.0227,0.0000,0.0227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11824,0.0556,0.0000,0.0556,0.0556,0.0000,0.0278,0.0278,0.0833,0.0556,0.0556,0.0000,0.0556,0.0556,0.0556,0.0278,0.1111,0.0278,0.1944,0.0278,0.0278
11825,0.0000,0.1786,0.1071,0.1071,0.0357,0.0714,0.0357,0.0357,0.0714,0.0000,0.0000,0.1071,0.0000,0.0714,0.0357,0.0714,0.0357,0.0000,0.0357,0.0000
11826,0.0870,0.0145,0.0580,0.1014,0.0290,0.0145,0.0000,0.0725,0.0725,0.1159,0.0580,0.0000,0.0725,0.0725,0.0435,0.1014,0.0435,0.0290,0.0000,0.0145
11827,0.0000,0.0000,0.2000,0.0000,0.0000,0.0000,0.1000,0.1000,0.1000,0.1000,0.0000,0.0000,0.0000,0.1000,0.0000,0.1000,0.0000,0.2000,0.0000,0.0000


In [ ]:
# add the features for the binding regions to the combined dataframe
df_binding_region_combined["amino_acid_composition"] = df_binding_region_combined["binding_region_seq"].apply(get_amino_acid_composition)
df_binding_region_combined["scd"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_scd(x, need_rounding=True, round_to=4))
df_binding_region_combined["shd"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_shd(x, need_rounding=True, round_to=4))
df_binding_region_combined["fcr"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_fcr(x, need_rounding=True, round_to=4))
df_binding_region_combined["sec_str_freq"] = df_binding_region_combined["sec"].apply(get_sec_str_freq)
df_binding_region_combined["sec_str3_freq"] = df_binding_region_combined["sec"].apply(lambda x: get_sec_str_freq(x, convert_3_state=True))
df_binding_region_combined["avg_plddt"] = df_binding_region_combined["plddt"].map(np.mean).round(4)
df_binding_region_combined["avg_relasa"] = df_binding_region_combined["relasa"].map(np.mean).round(4)

# print the first 5 rows of the combined dataframe with features
print(df_binding_region_combined.head())

# save the combined dataframe with features as a csv file in the processed data folder
df_binding_region_combined.to_csv(processed_data_folder / "binding_region_combined_features.csv", index=False)

# save the combined dataframe with features as a pickle file in the processed data folder
#df_binding_region_combined.to_pickle(processed_data_folder / "binding_region_combined_features.pkl")

  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                                sec  \
0       32  [C, C, H, H, H, H, H, H, H, H, H, H, H, H, H, ...   
1      214         [H, H, H, H, H, H, T, G, G, G, C, C, H, H]   
2       67  [C, C, P, P, P, P, P, P, P, G, G, G, S, C, H, ...   
3      449  [H, H, H, H, H, H, H, H, H, H, H, H, H, H, H, ...   
4      104  [C, C, C, C, C, C, C, C, C, C, C, C, C, C, C, ...   

                                              relasa  \
0  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2 

In [ ]:
# expand the features
df_aa = pd.DataFrame(df_binding_region_combined["amino_acid_composition"].tolist(), index=df_binding_region_combined.index).add_prefix("aa_freq_")
df_ss = pd.DataFrame(df_binding_region_combined["sec_str_freq"].tolist(), index=df_binding_region_combined.index).add_prefix("ss_freq_")
df_ss3 = pd.DataFrame(df_binding_region_combined["sec_str3_freq"].tolist(), index=df_binding_region_combined.index).add_prefix("ss3_freq_")
df_final = pd.concat([df_binding_region_combined.drop(columns=["amino_acid_composition", "sec_str_freq", "sec_str3_freq"]), df_aa, df_ss, df_ss3], axis=1)

print(df_final.head())

# save the final dataframe with expanded features as a pickle file in the processed data folder
df_final.to_pickle(processed_data_folder / "binding_region_combined_features_expanded.pkl")


  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                                sec  \
0       32  [C, C, H, H, H, H, H, H, H, H, H, H, H, H, H, ...   
1      214         [H, H, H, H, H, H, T, G, G, G, C, C, H, H]   
2       67  [C, C, P, P, P, P, P, P, P, G, G, G, S, C, H, ...   
3      449  [H, H, H, H, H, H, H, H, H, H, H, H, H, H, H, ...   
4      104  [C, C, C, C, C, C, C, C, C, C, C, C, C, C, C, ...   

                                              relasa  \
0  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2 

In [ ]:
#df_dd_do_seq["id_col"] = df_dd_do_seq["uniprot_id"] + "_" + df_dd_do_seq["start_pos"] + "_" + df_dd_do_seq["end_pos"]
#df_dd_do_seq

,uniprot_id,binding_region_seq,start_pos,end_pos,label,id_col
0,Q04917,QQDEEAGEGN,236,246,disorder_to_disorder,Q04917_236_246
1,P62258,QGDGEEQNKEALQDVEDENQ,235,255,disorder_to_disorder,P62258_235_255
2,P31947,NAGEEGGEAPQEPQS,233,248,disorder_to_disorder,P31947_233_248
3,P27348,SAGEECDAAEGAEN,231,245,disorder_to_disorder,P27348_231_245
4,P63104,QGDEAEAGEGGEN,232,245,disorder_to_disorder,P63104_232_245
...,...,...,...,...,...,...
11765,A0A0A0MTJ0,NQKKIRGILESLMCNESS,287,305,disorder_to_order,A0A0A0MTJ0_287_305
11766,A0A0A0MTJ0,ETLQAFDSHYDYTICGDSED,375,395,disorder_to_order,A0A0A0MTJ0_375_395
11767,U3KQA6,EQELVVTIPKN,183,194,disorder_to_order,U3KQA6_183_194
11768,A0A140T913,MTHHAVSDHEATL,212,225,disorder_to_order,A0A140T913_212_225


In [ ]:
#from df_to_fasta import dataframe_to_fasta
#dataframe_to_fasta(df_dd_do_seq, "id_col", "binding_region_seq", "disorder_to_disorder_and_disorder_to_order_sequences.fasta")
# filter same sequences
#df_dd_do_seq.drop_duplicates(subset=["binding_region_seq"], inplace=True)
#print(f"Number of unique sequences in disorder to disorder and disorder to order binding mode classes: {len(df_dd_do_seq)}")

Number of unique sequences in disorder to disorder and disorder to order binding mode classes: 11762
